In [ ]:
#****Note: it is required to have an input file for the polygenic risk score from the discovery GWAS*****
# We have included an example of a properly formatted SBAYESRC output file:
# 'workspace/practical_considerations_bucket/mhdata/height_prs.tsv'

In [ ]:
# !pip install polars
# !pip install --upgrade pandas-gbq
# !pip install --upgrade pandas google-cloud-bigquery
# !pip install "numpy<2.0.0"
# !pip install --upgrade fsspec

In [ ]:
# !pip install AoUPRS

In [ ]:
import fsspec

In [ ]:
import os
import json
import re
import subprocess
import numpy as np
import pandas as pd
import polars as pl
from google.cloud import bigquery
import pandas_gbq
import warnings

In [ ]:
#import hail
import AoUPRS
from datetime import datetime
import gcsfs
import multiprocessing
import ast
import concurrent.futures
import glob
import hail as hl
import time
import datetime
import matplotlib.pyplot as plt
from scipy.stats import norm

In [ ]:
#setup of variables
def wb(*args):
    """Run a wb command and return parsed JSON."""
    cmd = ["wb", *args, "--format=json"]
    result = subprocess.check_output(cmd, text=True)
    return json.loads(result)
 
 
# Get workspace info
workspace = wb("workspace", "describe")
GOOGLE_CLOUD_PROJECT = workspace["googleProjectId"]
 
# Get resources
resources = wb("resource", "list")
 
# WORKSPACE_BUCKET
bucket_resources = [
    r for r in resources
    if r.get("resourceType") == "GCS_BUCKET"
    and "practical_considerations_bucket" in r.get("id", "")
    and "temporary" not in r.get("id", "")
]
 
if not bucket_resources:
    raise ValueError("No matching bucket found")
 
WORKSPACE_BUCKET = f"gs://{bucket_resources[0]['bucketName']}"
 
# WORKSPACE_CDR
bq_resources = [
    r for r in resources
    if r.get("resourceType") in {"BQ_DATASET", "BIGQUERY_DATASET"}
]
 
cdr_resources = [
    r for r in bq_resources
    if re.match(r"^C\d{4}Q\d+R\d+$", r.get("datasetId", ""))
]
 
if not cdr_resources:
    raise ValueError("No matching CDR dataset found")
 
WORKSPACE_CDR = (
    f"{cdr_resources[0]['projectId']}."
    f"{cdr_resources[0]['datasetId']}"
)


In [ ]:
#get variables
version = WORKSPACE_CDR
bucket = WORKSPACE_BUCKET
cohort = "allofus"
google_project_id = GOOGLE_CLOUD_PROJECT

In [ ]:
os.environ["WORKSPACE_CDR"] = version

In [ ]:
google_project_id = %env GOOGLE_CLOUD_PROJECT

In [ ]:
#set working directory to be within mounted bucket
#this means that reading and writing will now directly read from and write to the bucket
os.chdir("/home/dataproc/workspace/practical_considerations_bucket/mhdata")
!pwd

In [ ]:
hl.init(
    gcs_requester_pays_configuration=google_project_id, default_reference='GRCh38')

In [ ]:
# Get the current date and time
start_time = datetime.datetime.now()

# Record the start time
current_date = start_time.date()
current_time = start_time.time()

# Format the current date
formatted_start_date = current_date.strftime("%Y-%m-%d")

# Format the current time
formatted_start_time = current_time.strftime("%H:%M:%S")

# Print the formatted date and time separately
print("Start date:", formatted_start_date)
print("Start time:", formatted_start_time)

Read Hail VDS

In [ ]:
srWGS_snpindel_bucket = 'gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel'
vds_srwgs_bucket_path = f'{srWGS_snpindel_bucket}/vds/hail.vds'
vds_srwgs_bucket_path

In [ ]:
vds = hl.vds.read_vds(vds_srwgs_bucket_path)

In [ ]:
vds.n_samples()

Drop Flagged srWGS samples for relatedness

In [ ]:
# Read flagged related samples
flagged_samples_path = "gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/relatedness/relatedness_flagged_samples.tsv"
!gsutil -u $$GOOGLE_PROJECT cat $flagged_samples_path > flagged_samples.cvs
# Import flagged samples into a hail table
flagged_samples = hl.import_table(flagged_samples_path, key='sample_id')
# Drop flagged sample from main Hail VDS
vds_no_flag = hl.vds.filter_samples(vds, flagged_samples, keep=False)
vds_no_flag.n_samples()

Define Sample Intended for PRS Calculation

In [ ]:
#select participants with both WGS and EHR data
dataset_16967016_person_sql = """
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id  
/*    
    WHERE
        person.PERSON_ID IN (
            SELECT
                distinct person_id  
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
            WHERE

                cb_search_person.person_id IN (
                    SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_ehr_data = 1 
                ) 
           
                cb_search_person.person_id IN (
                    SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 
                ) 
            )*/"""

dataset_16967016_person_df = pandas_gbq.read_gbq(
    dataset_16967016_person_sql,
     dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
     progress_bar_type="tqdm_notebook"
)

In [ ]:
dataset_16967016_person_df['person_id'].nunique()

In [ ]:
#subset to people in the auxilary no flag
vds_samples = vds_no_flag.variant_data.col_key.collect()
vds_sample_ids = set([int(row.s) for row in vds_samples])
person_in_vds_df = dataset_16967016_person_df[
    dataset_16967016_person_df['person_id'].isin(vds_sample_ids)
]

print(f"Original dataframe: {len(dataset_16967016_person_df)} people")
print(f"After VDS subset: {len(person_in_vds_df)} people")


In [ ]:
unique_ids = person_in_vds_df['person_id'].unique()
allofus_id = pd.DataFrame(unique_ids, columns=['person_id'])

In [ ]:
# #save locally
# allofus_id.to_csv('people_with_WGS_EHR_ids.csv', index=False)

In [ ]:
# save to the bucket
!gsutil cp people_with_WGS_EHR_ids.csv {bucket}/mhdata/prs_calculator_tutorial/prs_calculator_hail_vds/

In [ ]:
# can start from here if already saved to bucket previously
sample_needed_ht = hl.import_table(f'{bucket}/mhdata/prs_calculator_tutorial/prs_calculator_hail_vds/people_with_WGS_EHR_ids.csv', delimiter=',', key='person_id')

In [ ]:
# Filter samples
vds_subset = hl.vds.filter_samples(vds_no_flag, sample_needed_ht, keep=True)

In [ ]:
vds_subset.n_samples()

Prepare PRS Weight Table

In [ ]:
df = pl.read_csv(
    "height_prs.tsv",
    separator="\t"
)

In [ ]:
## example of how prs file should be formatted:
# cols include chr, bp, effect_allele, noneffect_allele, and weight for each snp in the model
print(df.shape)
print(df.columns)
print(df.head())

In [ ]:
df = df.with_columns([
    pl.col("chr").cast(pl.Int64),
    pl.col("bp").cast(pl.Int64),
    pl.col("weight").cast(pl.Float64),
    pl.col("effect_allele").str.to_uppercase(),
    pl.col("noneffect_allele").str.to_uppercase(),
])

In [ ]:
df = df.drop_nulls(subset=["bp"])

# 2. Cast the 'bp' column to Int64 (this removes the .0 decimals)
df = df.with_columns(
    pl.col("bp").cast(pl.Int64)
)

# 3. Verify the data type is now Int64
print("Updated Polars Schema:")
print(df.select("bp").schema)

# 4. Show the first few rows to ensure they are integers
print(df.select("bp").head())

In [ ]:
#save file to bucket
df.write_csv("Height_PRS_for_scoring.csv")

In [ ]:
AoUPRS.prepare_prs_table("mhdata/Height_PRS_for_scoring.csv",
                 "mhdata/Height_PRS_weighttable.csv", bucket=bucket)

In [ ]:
with gcsfs.GCSFileSystem().open(f'{bucket}/mhdata/Height_PRS_weighttable.csv', 'rb') as gcs_file:
    Height_weights_table = pd.read_csv(gcs_file)

In [ ]:
Height_weights_table.shape

In [ ]:
Height_weights_table.head()

PRS Calculator

In [ ]:
prs_identifier = 'Height'
pgs_weight_path = f'{bucket}/mhdata/Height_PRS_weighttable.csv'
output_path = f'{bucket}/mhdata/Height_scores'

In [ ]:
#monkeypatch pandas
if not hasattr(pd.errors, 'SettingWithCopyWarning'):
    pd.errors.SettingWithCopyWarning = FutureWarning


In [ ]:
AoUPRS.calculate_prs_vds(
    vds_subset,
    prs_identifier="Height",
    pgs_weight_path='mhdata/Height_PRS_weighttable.csv',
    output_path="mhdata/Height_output",
    bucket=bucket,
    save_found_variants=False,
    chunk_size=50000  # recommended for v9 stability
)

In [ ]:
#produce a dotplot
df = pd.read_csv(f"{bucket}/mhdata/Height_output/score/Height_final_scores.csv")

In [ ]:
#save in bucket
df = df.rename(columns={"sample_id":"person_id"})

In [ ]:
#save to bucket
destination_filename = "Height_final_scores_pids.csv"
local_path = destination_filename  # saves in current directory

df.to_csv(local_path)

gcs_path = f"{bucket}/mhdata/{destination_filename}".replace("//", "/").replace("gs:/", "gs://")

result = subprocess.run(
    ["gsutil", "cp", local_path, gcs_path],
    capture_output=True,
    text=True,
    check=False,
)

In [ ]:
# plt.figure(figsize=(6, 4))
# plt.scatter(range(len(df)), df["sum_weights_scaled"], s=10)
# plt.xlabel("Index")
# plt.ylabel("Scaled PRS (z-score)")
# plt.title("Scaled polygenic risk score")
# plt.show()

df_sorted = df.sort_values("sum_weights")
plt.scatter(range(len(df_sorted)), df_sorted["sum_weights"])
plt.xlabel("Individuals (sorted)")

In [ ]:
# Data
x = df["sum_weights"].dropna()

# Plot histogram (density=True makes it comparable to a PDF)
plt.figure(figsize=(6, 4))
plt.hist(x, bins=40, density=True, alpha=0.6)

# Normal distribution curve
mu, sigma = x.mean(), x.std()
xmin, xmax = plt.xlim()
xs = np.linspace(xmin, xmax, 200)
plt.plot(xs, norm.pdf(xs, mu, sigma))

plt.xlabel("PRS (z-score)")
plt.ylabel("Density")
plt.title("Distribution of polygenic risk score")
plt.show()
